# Comparison of our inertial theory with the experiments reported in Zeff et al. (2000) and Cattaneo et al. (2025)

(C) The fabulous self-similar Worthington jets team, September 2026

In [ ]:
# Include libraries

from numpy import *
from scipy import integrate
from scipy import optimize
from scipy import interpolate
from scipy import special
from scipy import io

# Use the notebook default plotting backend for cross-platform execution.
import matplotlib.pyplot as plt
import matplotlib

plt.rcParams.update({
    "font.family": "serif",
    "text.usetex": True,
    "font.size": 14,
    "axes.labelsize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 13,
})


In [ ]:
################################################################################################
#   def compute_nu_alpha(beta)
#
#   Computes the value of nu and alpha for a given value of beta (expressed in rad)
#
#   Usage:
#       nu, alpha = compute_nu_alpha(beta)
################################################################################################
def compute_nu_alpha(beta):

    #
    #   Determine nu and alpha
    #
    nu0 = 0
    x   = cos(pi-beta)
    nu = optimize.fsolve(lambda nu:special.hyp2f1(nu+1, -nu, 1, 0.5*(1-x)), nu0)[0]

    alpha = 1/(2-nu)

    return nu, alpha

**Experiments by Lathrop's group (Nature, 2000)**

In [ ]:
# Value of beta extracted from Fig. 3 in Zeff et al. Nature (2000)
# beta = (90-37.20)*pi/180
# Value of beta provided by the authors from Fig. 3 in Zeff et al. 
beta = (60)*pi/180
nu, alpha = compute_nu_alpha(beta)

print('beta  = {:f}'.format(beta*180/pi))
print('nu    = {:g} \nalpha = {:g}'.format(nu, alpha))

In [ ]:
#
#   Data of lc vs. t0-t from Fig. 2
#
data_Fig2Zeff = io.loadmat('data/Fig2_Zeff_etal2000.mat', \
                           squeeze_me=True, simplify_cells=True)
t  = data_Fig2Zeff['t']
lc = data_Fig2Zeff['lc']

In [ ]:
plt.figure(1)
plt.clf()
plt.loglog(t, lc, 'o', label='Exps. Fig. 2c, Zeff et al. 2000')
plt.plot(t, 0.35*t**(2/3), '-', label='Inertio-capillary, $\\alpha = 2/3$')
plt.plot(t, 0.35*t**alpha, '-', label='Inertial, $\\alpha = {:f}$'.format(alpha))
plt.xlabel('$t-t_0$ [ms]', fontsize=18)
plt.ylabel('$\\ell_c$ [cm]', fontsize=18)
plt.legend()
plt.grid(True)

In [ ]:
# Value of beta extracted from Fig. in Zeff et al. Nature (2000)
beta = (90-31)*pi/180   # The figure is inclined!
nu, alpha = compute_nu_alpha(beta)

print('beta  = {:f}'.format(beta*180/pi))
print('nu    = {:g} \nalpha = {:g}'.format(nu, alpha))

In [ ]:
#
#   Data of jet velocity from Fig. 4
#
data_Fig4Zeff = loadtxt('data/Fig4_Zeff_etal2000.dat')
h = data_Fig4Zeff[:,0]      # h [cm]
u = data_Fig4Zeff[:,1]      # u [m/s]

hc = 6.15    # Critical height, manually obtained from Fig. 4, Zeff et al.

hthe = linspace(min(h), max(h), 1000)
ugrs = 0.7*(1e-2*abs(hthe-hc))**((alpha-1)/alpha)

In [ ]:
plt.figure(2)
plt.clf()
plt.plot(h, u, 'o', label='Exps. Fig. 4, Zeff et al. 2000')
plt.plot(hthe, ugrs, '-', label='Inertial, $\\alpha = {:f}$'.format(alpha))
plt.xlabel('$h$ [cm]', fontsize=18)
plt.ylabel('$u$ [m/s]', fontsize=18)
plt.ylim((0, 60))
plt.legend()
plt.grid(True)

**Experiments by Supponen's group (PRL, 2025)**

In [ ]:
# Value of beta extracted from Fig. 5a in Cattaneo et al. PRL (2025)
beta = 46*pi/180
nu, alpha = compute_nu_alpha(beta)

print('beta  = {:f}'.format(beta*180/pi))
print('nu    = {:g} \nalpha = {:g}'.format(nu, alpha))

In [ ]:
data_Cattaneo = loadtxt('data/Fig5a_Cattaneo2026.dat')
h = data_Cattaneo[:,0]      # h [µm]
u = data_Cattaneo[:,1]      # u [m/s]

hc = 20  # Value taken manually from Fig. 5a in Cattaneo et al. (2025)

rho   = 1e3                                     # Water density [kg/m**3]
sigma = 0.072                                   # Surface tension [N/m]
mu    = 1e-3                                    # Dynamics viscosity [Pa.s]

Ohc = mu/sqrt(rho*sigma*hc*1e-6)                     # Oh based on the critical hc

Vc  = sigma/mu*Ohc**(-2*(alpha-1)/alpha)

# Reproducing the analytical solution of Cattaneo et al.
omega = 2*pi*100e3/2                            # The frequency of the Faraday wave 
                                                # is half that of the forcing [rad**-1]
lamb  = 2*pi*(sigma/(rho*omega**2))**(1/3)      # Dispersion relation

hthe = linspace(min(h), max(h), 1000)
uthe = sqrt((1e-6*hthe)**2/(1e-6*abs(hthe-hc))*sigma/rho*(2*pi)**3/lamb**2)

# This is our theoretical prediction, with the prefacor C = 4.6 fitted manually
C = 4.6
ugrs = C*Vc*(abs(hthe-hc)/hc)**((alpha-1)/alpha)


In [ ]:
#
#   Plot using the style of the figures by Vatsal
#

fig, ax = plt.subplots(figsize=(5, 6))

# Lines styled to match the image
ax.plot(h, u, 'o', label='Exps. Fig. 5a Cattaneo et al. (2025)')
ax.plot(hthe, uthe, '--', color='gray', linewidth=2, label=r'Eq. (6) Cattaneo et al. (2025)')
ax.plot(hthe, ugrs, '-', color='black', linewidth=2, label='Inertial solution, $\\alpha = {:0.3f}$'.format(alpha))

# Axis labels & subplot tag
ax.set_xlabel(r'$h$ [µm]')
ax.set_ylabel(r'$V_j$ [m/s]')
# ax.text(0.04, 0.93, r'(b)', transform=ax.transAxes, fontsize=20)

# Inward ticks on all 4 borders & bold spines
ax.tick_params(direction='in', which='both', top=True, right=True, length=5, width=1)
for spine in ax.spines.values():
    spine.set_linewidth(1.2)

ax.set_ylim(0, 80)

# Legend positioned below the plot without frame
ax.legend(loc='upper left', bbox_to_anchor=(-0.02, -0.15), frameon=False, handlelength=1.8)

plt.tight_layout()
plt.show()

In [ ]:
# Print the values of the Weber number, Oh and Vc to make sure we are on the right regime
We = rho*u**2*(1e-6*h)/sigma
print('We = ', We)
print('Oh = ', Ohc)
print('Vc = ', Vc, ' m/s')
